# Regressing the mid-price change on the imbalances

Two regressions of the same two statistics, and the whole of this notebook is the
distinction between them.

The **contemporaneous** regression puts the change in the mid-price over $(t-w,\,t]$ on the
order flow imbalance accumulated over the *same* interval,
$$\Delta P^m_{(t-w,\,t]} \;=\; a + b\,\mathrm{OFI}_{t,w} + \varepsilon .$$
This is the regression of Cont, Kukanov and Stoikov (2014), and it is a statement about
**price formation**: the flow that moved the price is on the right-hand side, and the
question is how tightly the mechanics tie the two together.

The **predictive** regression puts the change over $(t,\,t+h]$ on statistics measured at
$t$,
$$\Delta P^m_{(t,\,t+h]} \;=\; a + b\,\mathrm{OFI}_{t,w} + c\,I^n_t + \varepsilon .$$
This is a statement about **forecasting**, and nothing mechanical connects the two sides.
The first regression can have an $R^2$ near one and tell us nothing about the second.

Everything here is synthetic. The generator is the multivariate Hawkes flow of
`unito26.lob.hawkes` driving the marks of `unito26.lob.simulate`, and no parameter is
chosen, defended or sanity-checked against recorded data. The point is not to describe a
market; it is to ask what these two estimators do in a laboratory where the answer is known.

The theory is in
[`documentation/point-processes-and-hawkes.md`](../documentation/point-processes-and-hawkes.md),
and the confirmatory claims were frozen in
[`documentation/pre-registration-imbalance-regression.md`](../documentation/pre-registration-imbalance-regression.md)
before the first confirmatory run. This notebook cites both rather than restating them.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from scipy.linalg import expm, solve_lyapunov

from unito26.lob import config, estimation as est, imbalance_regression as ir
from unito26.lob.benchmark import REFERENCE_PRICE
from unito26.lob.hawkes import (
    ExponentialHawkes, HawkesParams, intensities_at_events, with_cross_pressure_scaled,
)
from unito26.lob.messages import (
    BUY, SELL, GridDepth, Horizon, ReportedDepth, SweepSize, Window,
)
from unito26.lob.orderbook import AggregateBook
from unito26.lob.session import MarketSession
from unito26.lob.simulate import EventType, MarkParams, OrderFlowSimulator
from unito26.lob.statistics import SessionStatistics

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
pio.templates["unito26"] = go.layout.Template(layout=dict(
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, font=dict(color=INK, size=12),
    xaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
    yaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
))
pio.templates.default = "unito26"
pd.set_option("display.width", 160)

#: The operating point.  Nothing is cached between runs: `data/` is not in the repository,
#: and every number below is produced by the cell that reports it.
WARM_UP, SAMPLE = 4800.0, 8400.0        # absolute horizons: the sample is (4800, 8400]
DEPTH = ReportedDepth(10)
LEVELS = tuple(GridDepth(n) for n in (1, 2, 5))
SPEC = SessionStatistics(LEVELS, (SweepSize(100),), (1, 10))
PRICE_UNIT = 100
PRESSURE = np.array([e.pressure for e in EventType], dtype=float)

#: Twelve is the pre-registered confirmatory block.  Lower it to read the notebook through
#: quickly; every reported figure below was produced at twelve.
SEEDS = 12
EXPLORATORY = range(0, 30)
CONFIRMATORY = range(1000, 1000 + SEEDS)

FLOW, MARKS = config.example_order_flow_params(), config.example_mark_params()
print(f"rho = {FLOW.branching_ratio:.4f}   nu = {FLOW.stationary_intensity().sum():.4f}/s"
      f"   beta = {FLOW.decay:.0f}/s   marks = {MARKS}")

rho = 0.6000   nu = 30.1878/s   beta = 60/s   marks = MarkParams(depth_decay=0.15, mean_log_size=4.0, sigma_log_size=0.8, lot=10)


## 2. What the theory gives us

The reference derives four things this notebook uses, and they are summarised here in a
paragraph each rather than re-derived.

**The filtration is the subject.** An intensity is defined relative to a filtration and is
meaningless without one. The simulator makes this concrete: a withdrawal on an empty side is
dropped, so the message stream is a state-dependent *thinning* of the point process and an
observer of messages knows strictly less than the Hawkes state does. `EventJournal` holds
the difference.

**Adaptedness, not predictability, is the alignment rule.** Predictability is a requirement
on the *compensator*. A regressor need only be $\mathcal{F}_t$-adapted, and
$\mathrm{OFI}_{t,w}$ contains $e_n$, so it is adapted and not predictable — a rule demanding
predictability of it would forbid this notebook's own statistic. What the distinction
governs is **ties**: with several rows at one timestamp, adaptedness decides which of them
the regressor reads and which the outcome starts from.

**$\mathrm{OFI}$ is a misspecified filter on the state, not a mistuned one.** The state
$S(t)$ is sufficient for the future of the flow, and the flow-only predictor of the next
event's pressure is built from the single scalar $\kappa = p^\top A\,S$.
$\mathrm{OFI}_{t,w}$ is a causal *boxcar* filter on that same state, which is what makes
tuning $w$ a question with an answer. It differs from $\kappa$ in three separable ways — the
window shape, the type weights, and the marks — and §8 measures each.

**The flow-only oracle is not a ceiling.** $\mathrm{sign}(\kappa)$ is optimal among
flow-only predictors. $I^n$ reads the book, which is a function of the marks as well as of
the points, so it can beat the oracle; at $\rho = 0$ the oracle is constant and $I^n$ still
predicts.

What the notebook keeps for itself is what it *measures*: the mean response and the
second-order structure, both of which the scalar formulae get wrong here in the same way and
for the same reason.

In [2]:
A, beta, mu = FLOW.excitation, FLOW.decay, FLOW.baseline
Gamma, identity = FLOW.branching_matrix, np.eye(FLOW.dimension)
lam_star = FLOW.stationary_intensity()
nu = float(lam_star.sum())

spectrum = np.sort(np.linalg.eigvals(A - beta * identity).real)
print("spectrum of A - beta I :", np.round(spectrum, 3))
print(f"  Perron mode          : {-beta * (1 - FLOW.branching_ratio):.3f} = -beta(1-rho)")
print(f"  a mode at -beta?     : no -- det A = {np.linalg.det(A):.3e}, so A is non-singular,")
print( "                         and 1/beta is the mean lag of a direct child, not a mode")
print(f"  distinct rates       : {len(np.unique(np.round(spectrum, 6)))} of 6, the repeated one")
print( "                         semisimple, which is what rules out a t*exp(lt) term")

spectrum of A - beta I : [-48.596 -45.891 -44.923 -43.876 -43.876 -24.   ]
  Perron mode          : -24.000 = -beta(1-rho)
  a mode at -beta?     : no -- det A = 2.271e+07, so A is non-singular,
                         and 1/beta is the mean lag of a direct child, not a mode
  distinct rates       : 5 of 6, the repeated one
                         semisimple, which is what rules out a t*exp(lt) term


In [3]:
# Second order.  The Lyapunov equation is the exact statement; the textbook scalar formula
# is its constant-column-sum special case, and the column sums here are not constant.
V = solve_lyapunov(A - beta * identity, -np.diag(lam_star))
exact = float(np.ones(6) @ V @ np.ones(6))
formula = nu / (2 * beta * (1 - FLOW.branching_ratio))

forced = Gamma * (Gamma.sum(axis=0).mean() / Gamma.sum(axis=0))
lam_forced = np.linalg.solve(identity - forced, mu)
V_forced = solve_lyapunov(forced * beta - beta * identity, -np.diag(lam_forced))
rho_forced = float(np.max(np.abs(np.linalg.eigvals(forced))))

print(f"Var(1'S)  exact {exact:.4f}   formula nu/(2 beta (1-rho)) {formula:.4f}"
      f"   error {100 * (exact / formula - 1):.1f}%")
print(f"  with the column sums forced equal: {float(np.ones(6) @ V_forced @ np.ones(6)):.6f}"
      f" against {lam_forced.sum() / (2 * beta * (1 - rho_forced)):.6f}  <- the cause")
print()
print("Relative standard deviation, compared object by object:")
var_lam = float(np.ones(6) @ A @ V @ A.T @ np.ones(6))
print(f"  1'S     exact {np.sqrt(exact) / (nu / beta):.4f}   formula {np.sqrt(formula) / (nu / beta):.4f}")
print(f"  bar-lam exact {np.sqrt(var_lam) / nu:.4f}   formula {np.sqrt(formula) / (nu / beta):.4f}"
      f"   <- damped by the baseline, mubar = {mu.sum():.2f} of nu = {nu:.2f}")

Var(1'S)  exact 0.6612   formula nu/(2 beta (1-rho)) 0.6289   error 5.1%
  with the column sums forced equal: 1.566918 against 1.566918  <- the cause

Relative standard deviation, compared object by object:
  1'S     exact 1.6162   formula 1.5762
  bar-lam exact 1.0551   formula 1.5762   <- damped by the baseline, mubar = 12.81 of nu = 30.19


In [4]:
# The mean response is exact, not mean-field: d E[S]/dt = mu + (A - beta I) E[S].
# Its half-life is *not* ln2 / (beta (1-rho)), which is only the asymptotic rate.
grid = np.linspace(0.0, 0.4, 4001)
rows = []
for j, name in ((0, "market"), (2, "limit"), (4, "withdrawal")):
    unit = np.zeros(6); unit[j] = 1.0
    lam_resp = np.array([float(np.ones(6) @ A @ expm((A - beta * identity) * t) @ unit) for t in grid])
    s_resp = np.array([float(np.ones(6) @ expm((A - beta * identity) * t) @ unit) for t in grid])
    rows.append({
        "shock": name,
        "intensity half-life (ms)": 1000 * grid[np.argmax(lam_resp <= lam_resp[0] / 2)],
        "state first crossing (ms)": 1000 * grid[np.argmax(s_resp <= s_resp[0] / 2)],
        "state peak": s_resp.max(),
        "peak at (ms)": 1000 * grid[s_resp.argmax()],
    })
print(f"asymptotic ln2 / (beta (1-rho)) = {1000 * np.log(2) / (beta * (1 - FLOW.branching_ratio)):.1f} ms")
print("the market-order response *exceeds* it, so no bound holds in that direction;")
print("and the state response to a market shock rises before it falls, the market column")
print(f"sum of Gamma being {Gamma.sum(axis=0)[0]:.3f} > 1, so its figure is a first crossing.")
pd.DataFrame(rows).round(3)

asymptotic ln2 / (beta (1-rho)) = 28.9 ms
the market-order response *exceeds* it, so no bound holds in that direction;
and the state response to a market shock rises before it falls, the market column
sum of Gamma being 1.310 > 1, so its figure is a first crossing.


,shock,intensity half-life (ms),state first crossing (ms),state peak,peak at (ms)
0,market,33.8,64.2,1.082,9.9
1,limit,24.4,21.3,1.000,0.0
2,withdrawal,28.6,23.3,1.000,0.0


## 3. The two statistics

$I^n_t$ is a **stock**: the imbalance of the sizes resting on the first $n$ grid positions
of each side,
$$I^n_t = \frac{\sum_{i\le n} S^{b,i}_t - \sum_{i\le n} S^{a,i}_t}
               {\sum_{i\le n} S^{b,i}_t + \sum_{i\le n} S^{a,i}_t} \in [-1, 1],$$
bid-heavy positive. It is a function of one configuration.

$\mathrm{OFI}$ is the corresponding **flow**. Each event contributes
$$e_n = \mathbf{1}\{P^b_n \ge P^b_{n-1}\}S^b_n - \mathbf{1}\{P^b_n \le P^b_{n-1}\}S^b_{n-1}
      - \mathbf{1}\{P^a_n \le P^a_{n-1}\}S^a_n + \mathbf{1}\{P^a_n \ge P^a_{n-1}\}S^a_{n-1},$$
and $\mathrm{OFI}_{t,w} = \sum_{t-w < t_n \le t} e_n$.

The crucial point, and the one §6 turns into a measurement: **$e_n$ is a touch-size
increment, not $p_n \times \text{size}$.** A limit order resting behind the touch contributes
zero; a cancellation that empties the touch contributes a whole queue; a market order that
walks the book registers the *stale* best size on the far side. So
$\mathrm{OFI} = \sum p_n g_n$ with a book-dependent random gain $g_n$, and the gain is where
the statistic stops being a signed event count.

`EventType.pressure` is $p_n$ — the sign an event *intends* to contribute. It is the
direction for the four order types and its negation for the two withdrawals, since
withdrawing a bid takes size off the buy side.

In [5]:
print(f"{'type':>14} {'direction':>10} {'pressure':>9}")
for event in EventType:
    print(f"{event.name:>14} {event.direction:>10} {event.pressure:>9}")
print()
print("pressure differs from direction exactly on the two withdrawals, and every type")
print("intends one sign or the other -- realising it is another matter, which is section 6.")

          type  direction  pressure
    MARKET_BUY          1         1
   MARKET_SELL         -1        -1
     LIMIT_BUY          1         1
    LIMIT_SELL         -1        -1
  WITHDRAW_BUY          1        -1
 WITHDRAW_SELL         -1         1

pressure differs from direction exactly on the two withdrawals, and every type
intends one sign or the other -- realising it is another matter, which is section 6.


## 4. The book, and the three-cornered trade that fixes the operating point

Before any regression, the laboratory has to be one. Three properties are wanted, and it
turns out they cannot all be had.

### The book has a restoring force, and it is in the marks

The intensities do not read the book, so nothing in the *flow* stabilises the depth. The
stabiliser is in the mark layer: `simulate._withdrawal` caps a withdrawal at the size
actually resting, so a thin level loses less than the size drawn while a deep one loses all
of it. Removal is therefore **sublinear** in depth where addition is not, and a stationary
depth exists at every composition. What the composition sets is its *level*.

That is not obvious from the update rules, so it is measured: the mean change in total book
size, conditioned on the current size.

In [6]:
def simulate(flow, marks, seed, warm_up=WARM_UP, sample=SAMPLE, journal=None):
    """A warmed book, the opening state, and the messages of the sample window."""
    simulator = OrderFlowSimulator(flow, marks, REFERENCE_PRICE, rng=seed)
    book = AggregateBook()
    simulator.warm_up(book, horizon=warm_up, journal=journal)
    opening, messages = book.copy(), []
    for message in simulator.stream(book, horizon=sample, journal=journal):
        book.apply(message, record=False)
        messages.append(message)
    return opening, messages, book


def session_at(flow, marks, seed, **kwargs):
    opening, messages, _ = simulate(flow, marks, seed, **kwargs)
    return MarketSession.from_occupied_levels(opening, messages, DEPTH, SPEC, PRICE_UNIT, True)

In [7]:
simulator = OrderFlowSimulator(FLOW, MARKS, REFERENCE_PRICE, rng=0)
book = AggregateBook()
simulator.warm_up(book, horizon=WARM_UP, journal=None)
sizes, changes = [], []
for message in simulator.stream(book, horizon=WARM_UP + 1800.0, journal=None):
    before = sum(book.levels_map(BUY).values()) + sum(book.levels_map(SELL).values())
    book.apply(message, record=False)
    after = sum(book.levels_map(BUY).values()) + sum(book.levels_map(SELL).values())
    sizes.append(before); changes.append(after - before)

sizes, changes = np.array(sizes, float), np.array(changes, float)
edges = np.quantile(sizes, np.linspace(0, 1, 9))
octile = np.clip(np.searchsorted(edges[1:-1], sizes, side="right"), 0, 7)
drift = pd.DataFrame({
    "size from": [f"{edges[k]:8.0f}" for k in range(8)],
    "size to": [f"{edges[k + 1]:8.0f}" for k in range(8)],
    "rows": [int((octile == k).sum()) for k in range(8)],
    "mean change per event": [changes[octile == k].mean() for k in range(8)],
}).round(3)
print(f"overall drift per event {changes.mean():+.4f} over {len(changes)} events -- "
      "which on its own would say nothing")
drift

overall drift per event +0.3714 over 55170 events -- which on its own would say nothing


,size from,size to,rows,mean change per event
0,59900,68520,6878,1.684
1,68520,72112,6915,-0.894
2,72112,76500,6895,0.732
3,76500,79970,6886,0.989
4,79970,82630,6905,-1.066
5,82630,86720,6887,0.651
6,86720,98710,6902,1.305
7,98710,105590,6902,-0.420


The sign of the mean change flips as the book deepens. That is a restoring force, not a
drift, and it is why a stationary depth exists at all.

### The three corners

With the spread pinned at one tick — which a concentrated book gives — the mid can move
**only** by clearing a whole touch queue. But clearing a touch queue is also the first step
in emptying a side. So the two events are the same event's tail, and only occupied depth
*behind* the touch separates them. Three properties are therefore in direct conflict:

1. **concentration** — most arriving orders resting within two or three ticks, which is the
   regime Cont–Stoikov's top-of-book $\mathrm{OFI}$ is about;
2. **a book that does not empty**, without which $e_n$ is undefined and $P^m$ does not
   exist, so every estimand becomes conditional on a two-sided book and the missingness sits
   exactly on the depletion bursts where $\mathrm{OFI}$ should carry most signal;
3. **a mid that moves often enough to regress on**.

Sharpening the mark offset buys (1) and pays in (2) and (3), because it puts the arriving
flow on the touch queue and leaves nothing behind it. The measurement below is over twenty
hour-long sessions a row, and the column that decides it is the largest single-event move.

In [8]:
#: Measured over twenty seeds at 4,800s warm-up and an hour of sample, at the balanced
#: composition.  Reproduce by running `sweep_offsets` below; it takes about twenty minutes.
OFFSET_SWEEP = pd.DataFrame([
    #  decay  within 3 ticks  size in top 3   spread  1-tick   moves/h  empty rows  max |dP|
    (0.10, 0.27, 0.17, 1.63, 0.59, 6568, 0.0000, None),
    (0.12, 0.32, 0.20, 1.46, 0.66, 5720, 0.0018, 15.5),
    (0.15, 0.39, 0.23, 1.30, 0.75, 4653, 0.0026, 15.5),
    (0.20, 0.49, 0.30, 1.15, 0.86, 3127, 0.0077, 58.0),
    (0.25, 0.58, 0.39, 1.08, 0.93, 1876, 0.0208, 159.5),
    (0.30, 0.66, 0.48, 1.04, 0.97, 1107, 0.0211, None),
    (0.35, 0.73, 0.56, 1.01, 0.99, 457, 0.0314, None),
], columns=["depth_decay", "orders within 3 ticks", "size in top 3", "spread",
            "one-tick spread", "mid moves / hour", "empty rows %", "max |dP| (ticks)"])
OFFSET_SWEEP

,depth_decay,orders within 3 ticks,size in top 3,spread,one-tick spread,mid moves / hour,empty rows %,max |dP| (ticks)
0,0.10,0.27,0.17,1.63,0.59,6568,0.0000,NaN
1,0.12,0.32,0.20,1.46,0.66,5720,0.0018,15.5
2,0.15,0.39,0.23,1.30,0.75,4653,0.0026,15.5
3,0.20,0.49,0.30,1.15,0.86,3127,0.0077,58.0
4,0.25,0.58,0.39,1.08,0.93,1876,0.0208,159.5
5,0.30,0.66,0.48,1.04,0.97,1107,0.0211,NaN
6,0.35,0.73,0.56,1.01,0.99,457,0.0314,NaN


Reading the table down: concentration rises monotonically and everything else that matters
falls with it. At $0.25$ the book is 58% within three ticks — arguably "most orders in the
first two or three levels" — and a side that clears jumps **159 ticks**, which is the
pathology the whole exercise exists to avoid. At $0.15$ the same twenty seeds cap the
largest single-event move at $15.5$ ticks with a median across seeds of $2.0$, hold the
spread at $1.30$ with three rows in four at one tick, and leave a side empty on 26 rows in a
million.

**The operating point takes $0.15$**, and the compromise is stated rather than buried: 39%
of arriving limit orders rest within three ticks, not "most". A study of *deeper*
$\mathrm{OFI}$ — Cont, Cucuringu and Zhang's multi-level version — would want the sharper
end and a different treatment of the tail.

### The composition is exactly one, and the exactness is the content

$\lambda_L/(\lambda_M + \lambda_W) = 1$ is a boundary, not a preference. Below it the book is
consumed faster than replenished and empties; above it the resting depth grows through the
session, and the contemporaneous coefficient $1/(2\bar S)$ stops being a coefficient and
becomes a moving target. Raising the limit-order baselines further does remove the empties,
and cheaply on the spread — but it buys that with a depth rising 60% to 87% within a
session, which is the same corner under another name.

The baselines are not chosen directly. What is chosen is $\lambda^*$, and
$\mu = (I - \Gamma)\lambda^*$ is read off, admissible exactly where it is non-negative
componentwise. The market-order component binds, market orders being heavily excited and
small in share.

In [9]:
print(f"lambda*  {np.round(lam_star, 4)}   nu = {nu:.4f}/s")
print(f"mu       {np.round(mu, 4)}   mubar = {mu.sum():.4f}")
limit = lam_star[[EventType.LIMIT_BUY, EventType.LIMIT_SELL]].sum()
print(f"\nlambda_L / (lambda_M + lambda_W) = {limit / (nu - limit):.6f}")
print(f"min mu = {mu.min():.4f} on component {int(mu.argmin())} "
      f"({EventType(int(mu.argmin())).name}) -- the binding one, with room on both sides")
print(f"\nendogenous fraction 1 - mubar/nu = {FLOW.endogenous_fraction():.4f}"
      f"   (NOT rho = {FLOW.branching_ratio:.2f})")
print(f"mean cluster size, mu-weighted   = {FLOW.mean_cluster_size():.4f}"
      f"   (NOT 1/(1-rho) = {1 / (1 - FLOW.branching_ratio):.2f})")
print(f"  certified by mubar x size = {mu.sum() * FLOW.mean_cluster_size():.4f} = nu")
print(f"signed endogenous fraction       = {FLOW.signed_endogenous_fraction(PRESSURE):.4f}")

lambda*  [2.1979 2.1979 7.547  7.547  5.3491 5.3491]   nu = 30.1878/s
mu       [0.7728 0.7728 3.3973 3.3973 2.233  2.233 ]   mubar = 12.8063

lambda_L / (lambda_M + lambda_W) = 1.000000
min mu = 0.7728 on component 0 (MARKET_BUY) -- the binding one, with room on both sides

endogenous fraction 1 - mubar/nu = 0.5758   (NOT rho = 0.60)
mean cluster size, mu-weighted   = 2.3573   (NOT 1/(1-rho) = 2.50)
  certified by mubar x size = 30.1878 = nu
signed endogenous fraction       = 0.3030


In [10]:
#: The warm-up is set by the book's own transient, which is three orders of magnitude longer
#: than the intensity's.  20/(beta(1-rho)) is asserted as a floor and is nowhere near binding.
print(f"intensity relaxation 1/(beta(1-rho)) = {1 / (beta * (1 - FLOW.branching_ratio)) * 1000:.0f} ms;"
      f"  20x it = {20 / (beta * (1 - FLOW.branching_ratio)):.2f} s")
print(f"warm-up used                        = {WARM_UP:.0f} s")
print("The book reaches 95% of its plateau depth about 3,900 s from cold, so the whole")
print("sample sits on the plateau.  A wall-clock warm-up that was generous at one end of")
print("the beta ladder and short at the other would manufacture that ladder's shape.")
assert WARM_UP >= 20 / (beta * (1 - FLOW.branching_ratio))

intensity relaxation 1/(beta(1-rho)) = 42 ms;  20x it = 0.83 s
warm-up used                        = 4800 s
The book reaches 95% of its plateau depth about 3,900 s from cold, so the whole
sample sits on the plateau.  A wall-clock warm-up that was generous at one end of
the beta ladder and short at the other would manufacture that ladder's shape.


### The regime gate

Every unconditional estimand below rests on the book being two-sided, so that is asserted
rather than reported. The gate is a **rate**, not a zero: a rare event cannot be asserted
away from a small block of seeds, and an earlier draft of this study did exactly that — six
clean seeds and 654,000 messages were read as "never", and thirty seeds showed twelve of
them emptying.

In [11]:
gate = []
for seed in CONFIRMATORY:
    recorded = session_at(FLOW, MARKS, seed)
    mid = ir.aligned_mid_price(recorded)
    spread = recorded.stats["Spread"].to_numpy(dtype=float)
    depth = recorded.stats["TouchDepth"].to_numpy(dtype=float)
    finite = depth[np.isfinite(depth)]
    sixth = len(finite) // 6
    gate.append({
        "seed": seed,
        "rows": len(mid),
        "segments": int(mid.segments.max()) + 1,
        "empty rows": int((~np.isfinite(spread)).sum()),
        "occupied of 10": float(np.nanmean(
            recorded.stats[["BidOccupiedLevels", "AskOccupiedLevels"]].to_numpy(float))),
        "touch depth": finite.mean() / 2,
        "last sixth / first": finite[-sixth:].mean() / finite[:sixth].mean(),
    })
GATE = pd.DataFrame(gate)
empty_rate = GATE["empty rows"].sum() / GATE["rows"].sum()
print(f"{GATE['rows'].sum():,} rows over {len(GATE)} seeds; "
      f"{GATE['empty rows'].sum()} empty-side rows = {100 * empty_rate:.5f}%")
assert empty_rate < 1e-3, "out of regime: the estimands below are no longer unconditional"
assert (GATE["last sixth / first"].between(0.5, 2.0)).all(), "the depth is not stationary"
GATE.round(4)

1,305,409 rows over 12 seeds; 0 empty-side rows = 0.00000%


,seed,rows,segments,empty rows,occupied of 10,touch depth,last sixth / first
0,1000,109120,1,0,10.0,895.1786,0.6580
1,1001,107966,1,0,10.0,1397.2419,1.1640
2,1002,109699,1,0,10.0,1039.0568,1.1173
3,1003,107960,1,0,10.0,1017.0396,1.5671
4,1004,108029,1,0,10.0,756.6726,0.8402
5,1005,109536,1,0,10.0,1577.2473,0.6802
6,1006,109074,1,0,10.0,1112.7906,0.7108
7,1007,110014,1,0,10.0,606.8555,1.2314
8,1008,109601,1,0,10.0,1311.2650,1.0596
9,1009,108123,1,0,10.0,1028.3446,1.2043


## 5. What $\Delta P^m$ is, and what that does to $R^2$

%For Claude: It is good that we explain the issue of persoming a naive regression of Delta P^m on the order flow , and the naive 
% diagnostics is fauly. However, I feel that we are getting lost in it. After stating and measuring the problem, we should clearly explain what to do about it. Trace this back to the original paper by Cont and Stoikov first and then proposed something better if you can come up with it. 
% In the original paper, the time windows is shown large enough that the mid-price is already "zoomed-out" and hence is appeares more as the sample path of a brownina motion rather than the microscopic stepwise function. So On the one hand we could perform a regression with say half-an-hour time windows and see how this behaves. 
% Secondly we shoudl tackle the issue of actually evaluating how OFI and QI perform as estimators on a genuinely high-frequency dynamics. 
% we can introduce micro-price (i.e. mid price adjusted by qi at the level of the touch) and predict this. Or we can see how we predict the VWAP instead. This is one angle. The other angle shoudl be how to construct a statistical measure of the effect of OFI and QI on the mid-price move (so genuinely mid-price, not micro-price or VWAP), possibly by looking at the correlation or having some devise around causality. What do you think of using mutual information, or actually DIRECTED information? Another idea, how about we count the upawrds jumps of the mid price, and the downward jumps of the midprice in the prediction window [t, t+h] (so not justt P(t+h) - P(t)) and we regress this frequency / count onto OFI?

The mid-price lives on a **half-tick lattice**: an odd spread puts it on a half tick, an
even one on a whole tick. At the operating point the spread is one tick on three rows in
four, so the mid is mostly on half ticks — and it does not move at all on about 96% of
events.

That is the regime, not an obstacle to be parametrised away. Section 4 showed why: the only
parametrisations in which the mid moves often are the ones in which the book is spread thin,
and the only ones in which the book is concentrated are the ones in which it barely moves.
What follows is therefore a set of diagnostics *built for* a lattice outcome with a large
atom, and $R^2$ shown failing on its own terms.

Three numbers say why $R^2$ is the wrong summary here.

**The participation ratio** $\left(\sum x^2\right)^2 / \sum x^4$ says how many observations
a sum of squares is really built on: $n$ if the mass is spread evenly, $1$ if one row
carries it.

**The Hill tail index** estimates the tail exponent. Below 2 the variance does not exist,
and neither does the population quantity $R^2$ estimates.

**The nonparametric ceiling** — the $R^2$ of a binned conditional mean — bounds what *any*
function of the predictor can achieve. If it sits close to the linear fit, linearity was
never the constraint.

In [12]:
RECORDED = {seed: session_at(FLOW, MARKS, seed) for seed in CONFIRMATORY}
W_STAR = Window(0.030)       # section 12 derives this; used here so the sections agree
BINS, PRIOR = 6, 20.0


def frame_for(recorded, window=W_STAR, levels=GridDepth(1)):
    # Predictors, outcome and the common row set, for one session.
    mid = ir.aligned_mid_price(recorded)
    predictors = ir.predictors(recorded, window, levels)
    outcome = ir.forward_change_to_next_event(mid)
    keep = predictors.defined & np.isfinite(outcome)
    return predictors, outcome, keep, mid


predictors, outcome, keep, mid = frame_for(RECORDED[CONFIRMATORY[0]])
change = outcome[keep]
print(f"{keep.sum():,} scored rows of {len(keep):,}  ({100 * keep.mean():.3f}% survive)")
print(f"P(dP = 0) = {np.mean(change == 0):.4f}   values seen: "
      f"{sorted(set(np.abs(change[change != 0])))[:6]} ... max {np.abs(change).max()}")
print(f"half-tick values present: {bool(np.any(np.abs(change) % 1 == 0.5))}")

109,117 scored rows of 109,120  (99.997% survive)
P(dP = 0) = 0.9563   values seen: [np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0)] ... max 2.0
half-tick values present: True


In [13]:
diagnostics = []
for name, values in (("dP (outcome)", change),
                     ("OFI", predictors.order_flow_imbalance[keep]),
                     ("OFI - e_n", (predictors.order_flow_imbalance - predictors.last_contribution)[keep]),
                     ("e_n", predictors.last_contribution[keep]),
                     ("I^1", predictors.queue_imbalance[keep])):
    diagnostics.append({
        "series": name,
        "participation ratio": est.participation_ratio(values),
        "as a share of n": est.participation_ratio(values) / values.size,
        "Hill index (top 1%)": est.hill_tail_index(values, max(20, values.size // 100)),
    })
print("The regressors are at least as heavy-tailed as the outcome -- a cancellation at the")
print("touch contributes a whole queue -- so the tail problem is not confined to dP.")
pd.DataFrame(diagnostics).round(4)

The regressors are at least as heavy-tailed as the outcome -- a cancellation at the
touch contributes a whole queue -- so the tail problem is not confined to dP.


,series,participation ratio,as a share of n,Hill index (top 1%)
0,dP (outcome),2491.5978,0.0228,1.9734
1,OFI,7924.9433,0.0726,4.4843
2,OFI - e_n,6153.5585,0.0564,4.4003
3,e_n,2665.4085,0.0244,3.0360
4,I^1,76959.8676,0.7053,311.2148


In [14]:
def nonparametric_ceiling(x, y, bins):
    # R^2 of the binned conditional mean: what *any* function of x could reach.
    edges = np.unique(np.quantile(x, np.linspace(0, 1, bins + 1)[1:-1]))
    cell = np.searchsorted(edges, x, side="right")
    fitted = np.zeros_like(y, dtype=float)
    for c in np.unique(cell):
        fitted[cell == c] = y[cell == c].mean()
    centred = y - y.mean()
    return 1.0 - float((y - fitted) @ (y - fitted)) / float(centred @ centred)


rows = []
for name, values in (("OFI", predictors.order_flow_imbalance[keep]),
                     ("e_n", predictors.last_contribution[keep]),
                     ("I^1", predictors.queue_imbalance[keep]),
                     ("OFI / 2AD", predictors.normalised_imbalance[keep])):
    fit = est.ols(np.column_stack([np.ones(values.size), values]), change)
    ceiling = nonparametric_ceiling(values, change, 20)
    rows.append({"predictor": name, "OLS R^2": fit.r_squared,
                 "nonparametric ceiling": ceiling, "ratio": ceiling / fit.r_squared})
print("Where the ceiling is far above the linear fit, linearity was not the binding")
print("constraint: the directional relation is present and the magnitude relation is not.")
pd.DataFrame(rows).round(5)

Where the ceiling is far above the linear fit, linearity was not the binding
constraint: the directional relation is present and the magnitude relation is not.


,predictor,OLS R^2,nonparametric ceiling,ratio
0,OFI,0.00009,0.00043,4.80897
1,e_n,0.00012,0.00027,2.16421
2,I^1,0.00975,0.01122,1.15127
3,OFI / 2AD,0.00021,0.00041,1.98320


### The replacements

Two diagnostics are primary, and both are built for the lattice.

**$\mathrm{Cov}(\mathrm{sign}(x),\, \Delta P^m)$**, in ticks per event. The *covariance* and
not the raw product $\mathbb{E}[\mathrm{sign}(x)\,\Delta P^m]$: under independence the raw
product is $\mathbb{E}[\mathrm{sign}\,x]\,\mathbb{E}[\Delta P^m]$, zero only if a factor is,
and the model has no anchor for the price level — so per seed $\mathbb{E}[\Delta P^m]$ is a
random drift, and any predictor with a non-zero mean sign inherits it. It is a random effect
*across* seeds, so it widens every interval too. Reported **gross**: subtracting a
half-spread from a per-row quantity guarantees a negative number by arithmetic and models
nothing.

**Three-class log-score skill against climatology** — the lattice-aware analogue of $R^2$.
Bin the predictor into equal-count quantile bins, estimate $P(\text{down},
\text{flat}, \text{up})$ per bin on a training fold, and score out of fold against the same
fold's climatology. It uses the atom as *data* rather than discarding it, and it is immune
to the tail. Three things it is not, all handled in `estimation.ThreeClassModel`: it is not
bounded below without shrinkage; it is not zero under the null in finite sample, carrying an
expected penalty of about $k(J-1)/(2n_{\text{train}})$ nats; and it is comparable across
predictors only at a fixed bin count.

Secondary: Kendall's $\tau_b$, ties-corrected and scale-free — the statistic only, never
`scipy`'s p-value, whose i.i.d. null is anticonservative by orders of magnitude on
overlapping event-sampled rows.

**All of them on one row set, or none of them.** Two predictors evaluated on the rows each
happens to define are two samples answering two questions, and a reading in which they swap
places between metrics is the symptom of exactly that.

In [15]:
def score_all(recorded, window=W_STAR, levels=GridDepth(1)):
    # Every metric for every predictor, on one common row set.
    predictors, outcome, keep, mid = frame_for(recorded, window, levels)
    y = outcome[keep]
    classes = ir.outcome_classes(y)
    half = y.size // 2
    named = (("OFI", predictors.order_flow_imbalance), ("e_n", predictors.last_contribution),
             ("I^1", predictors.queue_imbalance), ("OFI / 2AD", predictors.normalised_imbalance))
    rows = []
    for name, values in named:
        x = values[keep]
        fit = est.ols(np.column_stack([np.ones(x.size), x]), y)
        accuracy, coverage, base = ir.sign_accuracy(x, y)
        rows.append({
            "predictor": name,
            "R^2": fit.r_squared,
            "participation": est.participation_ratio(x),
            "Cov(sign, dP)": ir.signed_covariance(x, y),
            "log-score skill": est.log_score_skill(
                x[:half], classes[:half], x[half:], classes[half:], BINS, PRIOR),
            "tau-b": est.kendall_tau_b(x, y),
            "sign accuracy": accuracy,
            "coverage": coverage,
            "base rate": base,
        })
    return pd.DataFrame(rows), keep.mean()


table, survival = score_all(RECORDED[CONFIRMATORY[0]])
print(f"one session, {survival:.5f} of rows surviving, all metrics on the same rows")
table.round(5)

one session, 0.99997 of rows surviving, all metrics on the same rows


,predictor,R^2,participation,"Cov(sign, dP)",log-score skill,tau-b,sign accuracy,coverage,base rate
0,OFI,0.00009,7924.94326,0.00145,0.00351,0.01682,0.55510,0.02711,0.50068
1,e_n,0.00012,2665.40851,0.00092,0.00132,0.01857,0.57917,0.01285,0.50071
2,I^1,0.00975,76959.86759,0.01075,0.02607,0.08746,0.71038,0.04326,0.50127
3,OFI / 2AD,0.00021,1506.99469,0.00145,0.00332,0.01844,0.55510,0.02711,0.50068


Read the row for $I^1$ against the row for $\mathrm{OFI}$. The book-reading statistic wins
on every metric here, and $R^2$ and the log-score skill agree on the ordering — which they
need not, and which is worth checking rather than assuming. The earlier reading in which
they disagreed turned out to be two row sets rather than two questions: $\mathrm{OFI}$ was
NaN wherever its window held an undefined $e_n$ while $I^n$ was a finite $\pm 1$ on a
one-sided book.

Note the coverage column. `sign accuracy` conditions on *both* signs being non-zero, so it
is computed on 3% or 4% of rows, and accuracies under different conditioning are not
comparable. That is why it is reported with its coverage and its base rate or not at all.

## 6. The contemporaneous regression, and why it is not an identity

Cont, Kukanov and Stoikov derive the mechanical relation inside a stylized model: over a
bucket in which the depth beyond the best is $D$ on both sides and arrivals occur only at
the best,
$$\Delta P = \tfrac12\left\lceil \frac{L^b - C^b - M^s}{D} \right\rceil
           - \tfrac12\left\lceil \frac{L^s - C^s - M^b}{D} \right\rceil,$$
so that
$$\Delta P = \frac{\mathrm{OFI}}{2D} + \varepsilon, \qquad
  \varepsilon = \tfrac12\big[(\lceil x\rceil - x) - (\lceil y\rceil - y)\big] \in (-\tfrac12, \tfrac12)\ \text{ticks}.$$

The bound on $\varepsilon$ holds independently of the bucket length. **The $R^2 \approx 1 -
\text{const}/\mathrm{Var}(\Delta P_k)$ that is usually read off it does not**, and this
notebook predicts the failure rather than discovering it.

$\mathrm{Var}(\varepsilon)$ is not a constant. It tends to $1/24$ tick$^2$ only once
$|\mathrm{OFI}|/D \gg 1$. At short buckets $|\mathrm{OFI}|/D \ll 1$, both ceilings are
constant, $\Delta P$ is constant, and $\varepsilon = -\mathrm{OFI}/(2D) + \text{const}$ is
*perfectly* correlated with the regressor. Two consequences, both stated in advance and both
in this study's own regime:

1. the $R^2(\Delta t)$ curve has a different shape from the claim at small $\Delta t$;
2. the OLS slope is **attenuated below** $1/(2D)$ there — which will otherwise be read as
   the depth normalisation failing.

One convention is worth a line. With $\lceil\cdot\rceil$ on both sides, an arbitrarily small
net one-sided flow moves the price a full tick. Truncation toward zero is the defensible
alternative; it leaves the half-tick bound unchanged and changes the model-implied atom at
zero. The factor 2 is the mid being an average of two *prices* each moving in whole ticks,
not an average of two queues.

Two of the derivation's hypotheses fail in this simulator: the depth $D$ is the level
*beyond* the best, for which `AverageDepth` is only a proxy, and arrivals here are not
confined to the best. So the relation is not a near-identity, and the control that says so
is to **exclude price-changing events from $\mathrm{OFI}$ and refit** — under a near-identity
the fit would barely move.

In [16]:
def contemporaneous(recorded, windows):
    mid = ir.aligned_mid_price(recorded)
    flow = ir.aligned_order_flow(recorded)
    rows = []
    for window in windows:
        imbalance = ir.backward_sum(flow, Window(window))
        moved = ir.backward_change(mid, Window(window))
        keep = np.isfinite(imbalance) & np.isfinite(moved)
        x, y = imbalance[keep], moved[keep]
        fit = est.ols(np.column_stack([np.ones(x.size), x]), y)
        depth = np.nanmean(recorded.stats["TouchDepth"].to_numpy(float)) / 2
        rows.append({
            "window (s)": window, "rows": int(keep.sum()),
            "slope": fit.coefficients[1], "1 / (2 D)": 1 / (2 * depth),
            "slope / (1/2D)": fit.coefficients[1] * 2 * depth,
            "R^2": fit.r_squared, "mean |OFI| / D": np.mean(np.abs(x)) / depth,
        })
    return pd.DataFrame(rows)


WINDOWS = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
CONTEMPORANEOUS = contemporaneous(RECORDED[CONFIRMATORY[0]], WINDOWS)
print("The slope approaches 1/(2D) from below as the bucket lengthens and |OFI|/D grows,")
print("exactly as the attenuation argument predicts.  At the shortest buckets it is a")
print("fraction of the mechanical value and the R^2 is small -- not a failure of the")
print("normalisation but the regime where the rounding error *is* the regressor.")
CONTEMPORANEOUS.round(5)

The slope approaches 1/(2D) from below as the bucket lengthens and |OFI|/D grows,
exactly as the attenuation argument predicts.  At the shortest buckets it is a
fraction of the mechanical value and the R^2 is small -- not a failure of the
normalisation but the regime where the rounding error *is* the regressor.


,window (s),rows,slope,1 / (2 D),slope / (1/2D),R^2,mean |OFI| / D
0,0.01,109119,0.00079,0.00056,1.41959,0.09119,0.03727
1,0.03,109118,0.00080,0.00056,1.43511,0.10505,0.06051
2,0.10,109115,0.00079,0.00056,1.40722,0.12422,0.09839
3,0.30,109105,0.00075,0.00056,1.34824,0.12440,0.15155
4,1.00,109093,0.00070,0.00056,1.25279,0.14128,0.26990
5,3.00,109049,0.00069,0.00056,1.23576,0.20064,0.48385
6,10.00,108718,0.00056,0.00056,0.99812,0.25489,0.93306
7,30.00,108013,0.00040,0.00056,0.71084,0.29066,1.71591


In [17]:
figure = go.Figure()
figure.add_trace(go.Scatter(x=CONTEMPORANEOUS["window (s)"], y=CONTEMPORANEOUS["R^2"],
                            mode="lines+markers", name="R^2", line=dict(color=INK)))
figure.add_trace(go.Scatter(x=CONTEMPORANEOUS["window (s)"], y=CONTEMPORANEOUS["slope / (1/2D)"],
                            mode="lines+markers", name="slope / (1/2D)",
                            line=dict(color=MUTED, dash="dash"), yaxis="y2"))
figure.update_layout(
    title="The contemporaneous fit against bucket length",
    xaxis=dict(title="bucket length w (s)", type="log"),
    yaxis=dict(title="R^2"), yaxis2=dict(title="slope / (1/2D)", overlaying="y", side="right"),
    height=420, legend=dict(x=0.02, y=0.98))
figure.show()

In [18]:
# The control.  Under a near-identity, dropping the events that actually moved the price
# would gut the fit; if the relation is loose, it barely moves.
recorded = RECORDED[CONFIRMATORY[0]]
mid = ir.aligned_mid_price(recorded)
flow = ir.aligned_order_flow(recorded)
stepped = np.concatenate([[np.nan], np.diff(mid.values)])
quiet = ir.AlignedSeries(flow.times, np.where(stepped == 0, flow.values, 0.0), flow.segments)

rows = []
for label, series in (("all events", flow), ("price-changing events zeroed", quiet)):
    imbalance = ir.backward_sum(series, Window(1.0))
    moved = ir.backward_change(mid, Window(1.0))
    keep = np.isfinite(imbalance) & np.isfinite(moved) & np.isfinite(stepped)
    fit = est.ols(np.column_stack([np.ones(int(keep.sum())), imbalance[keep]]), moved[keep])
    rows.append({"OFI built from": label, "slope": fit.coefficients[1], "R^2": fit.r_squared})
print("A near-identity would collapse in the second row.  It does not collapse; it")
print("degrades, which is the honest description of the relation in this book.")
pd.DataFrame(rows).round(6)

A near-identity would collapse in the second row.  It does not collapse; it
degrades, which is the honest description of the relation in this book.


,OFI built from,slope,R^2
0,all events,0.000700,0.141279
1,price-changing events zeroed,0.000334,0.026841


### The per-event failure, exhibited

The mechanical relation is about buckets. Per *event* it fails outright, and the failure is
the substance of §3's warning about the gain $g_n$. Three events with the same $|\Delta P^m|$
of half a tick can have $e_n$ two orders of magnitude apart.

In [19]:
mid_values = ir.aligned_mid_price(recorded).values
contribution = ir.aligned_order_flow(recorded).values
step = np.concatenate([[np.nan], np.diff(mid_values)])
moved_half = np.isfinite(step) & (np.abs(step) == 0.5) & np.isfinite(contribution)
magnitudes = np.abs(contribution[moved_half])
magnitudes = magnitudes[magnitudes > 0]
print(f"events moving the mid by exactly half a tick: {moved_half.sum():,}")
print(f"  |e_n| quantiles  1% {np.quantile(magnitudes, 0.01):8.1f}"
      f"   50% {np.quantile(magnitudes, 0.50):8.1f}"
      f"   99% {np.quantile(magnitudes, 0.99):8.1f}")
print(f"  ratio of the 99th to the 1st percentile: {np.quantile(magnitudes, 0.99) / np.quantile(magnitudes, 0.01):.0f}x")
print()
print("Same price effect, |e_n| spanning two orders of magnitude.  A price-improving limit")
print("order contributes one lot; a cancellation emptying the touch contributes a whole")
print("queue; a walking market order registers only the stale far-side size.")

events moving the mid by exactly half a tick: 4,014
  |e_n| quantiles  1%     10.0   50%     50.0   99%    318.7
  ratio of the 99th to the 1st percentile: 32x

Same price effect, |e_n| spanning two orders of magnitude.  A price-improving limit
order contributes one lot; a cancellation emptying the touch contributes a whole
queue; a walking market order registers only the stale far-side size.


## 7. The predictive regression

Now the outcome moves to $(t,\,t+h]$ and nothing mechanical connects the two sides. The
shortest horizon there is is the **next event** — not a small duration, because an event
horizon does not shrink as the flow speeds up.

The alignment is where this section's risk sits, and it is worth being explicit. The base of
the forward difference is the row itself, by index, never a lookup on its timestamp: several
rows can share one clock reading and they are different states. The right edge at $t+h$ does
take its ties, because the outcome is measured *at* that instant. Getting either edge wrong
does not raise; it quietly turns this regression back into §6, which fits far better.

In [20]:
def predictive(recorded, window, horizons):
    mid = ir.aligned_mid_price(recorded)
    predictors = ir.predictors(recorded, window, GridDepth(1))
    rows = []
    for label, outcome in horizons(mid):
        keep = predictors.defined & np.isfinite(outcome)
        y = outcome[keep]
        classes = ir.outcome_classes(y)
        half = y.size // 2
        for name, values in (("OFI", predictors.order_flow_imbalance),
                             ("e_n", predictors.last_contribution),
                             ("I^1", predictors.queue_imbalance)):
            x = values[keep]
            fit = est.ols(np.column_stack([np.ones(x.size), x]), y)
            rows.append({
                "horizon": label, "predictor": name, "rows": int(keep.sum()),
                "P(dP=0)": float(np.mean(y == 0)),
                "R^2": fit.r_squared,
                "Cov(sign, dP)": ir.signed_covariance(x, y),
                "log-score skill": est.log_score_skill(
                    x[:half], classes[:half], x[half:], classes[half:], BINS, PRIOR),
                "tau-b": est.kendall_tau_b(x, y),
            })
    return pd.DataFrame(rows)


def horizon_set(mid):
    yield "next event", ir.forward_change_to_next_event(mid)
    for h in (0.01, 0.1, 1.0):
        yield f"{h} s", ir.forward_change(mid, Horizon(h))


PREDICTIVE = predictive(RECORDED[CONFIRMATORY[0]], W_STAR, horizon_set)
PREDICTIVE.round(5)

,horizon,predictor,rows,P(dP=0),R^2,"Cov(sign, dP)",log-score skill,tau-b
0,next event,OFI,109117,0.95633,0.00009,0.00145,0.00351,0.01682
1,next event,e_n,109117,0.95633,0.00012,0.00092,0.00132,0.01857
2,next event,I^1,109117,0.95633,0.00975,0.01075,0.02607,0.08746
3,0.01 s,OFI,53652,0.91609,0.00008,0.00152,0.00701,0.01261
4,0.01 s,e_n,53652,0.91609,0.00015,0.00234,0.00229,0.02323
5,0.01 s,I^1,53652,0.91609,0.01817,0.02248,0.02916,0.11817
6,0.1 s,OFI,98647,0.81598,0.00014,-0.00050,0.01031,0.00758
7,0.1 s,e_n,98647,0.81598,0.00003,0.00216,0.00291,0.01537
8,0.1 s,I^1,98647,0.81598,0.04059,0.05701,0.02581,0.15723
9,1.0 s,OFI,109117,0.55264,0.00087,-0.01495,0.00334,-0.01970


## 8. Why the fit is what it is: four rows and a ceiling

$\mathrm{OFI}$ is a *misspecified* filter on the state the intensity actually reads, and the
misspecification decomposes into three separable pieces. This section measures each by
building the intermediate predictors that isolate them.

* $\mathrm{OFI}$ — flat weights, boxcar window, book-dependent marks;
* the **type-weighted** boxcar — the same window, but each event weighted by
  $(p^\top A)_j$, which isolates what the flat weighting costs;
* the flow-only oracle $\mathrm{sign}(\kappa)$, $\kappa = p^\top A\,S(t^-)$ — the exponential
  window and the exact type weights, but no marks;
* flow **plus realised marks** — the actual ceiling.

The sanity identity is $p^\top \lambda^* = 0$, symmetry between the two directions. It is
also what the oracle's sign rule depends on: $\mathbb{P}(+) - \mathbb{P}(-)$ is $\kappa$
times a strictly positive integral, so the sign survives the shrinkage — but only because
$p^\top\mu = 0$, and on an asymmetric rung even that would fail.

In [21]:
kappa_weights = PRESSURE @ A
print("kappa weights p'A :", np.round(kappa_weights, 4))
print(f"market : limit weight ratio = {abs(kappa_weights[0] / kappa_weights[2]):.4f}")
print(f"p' lambda* = {float(PRESSURE @ lam_star):.2e}  <- the symmetry the oracle's sign rule needs")

kappa weights p'A : [ 30.2329 -30.2329  16.1242 -16.1242 -16.1242  16.1242]
market : limit weight ratio = 1.8750
p' lambda* = 8.88e-16  <- the symmetry the oracle's sign rule needs


In [22]:
def flow_only_comparison(seed, horizon_seconds, windows):
    # A Hawkes path with no book at all: the flow layer's own filters against kappa.
    simulator = ExponentialHawkes(FLOW, rng=seed)
    events = list(simulator.events(horizon_seconds))
    times = np.array([t for t, _ in events])
    kinds = np.array([k for _, k in events])
    kappa = intensities_at_events(FLOW, times, kinds) @ PRESSURE

    def boxcar(values, window):
        start = np.searchsorted(times, times - window, side="right")
        cumulative = np.concatenate([[0.0], np.cumsum(values)])
        return cumulative[1:] - cumulative[start]

    signs, weights = PRESSURE[kinds], kappa_weights[kinds]
    plain = np.array([np.corrcoef(boxcar(signs, w), kappa)[0, 1] for w in windows])
    typed = np.array([np.corrcoef(boxcar(weights, w), kappa)[0, 1] for w in windows])
    return times, kinds, kappa, plain, typed


WINDOW_GRID = np.exp(np.linspace(np.log(0.001), np.log(0.5), 40))
times, kinds, kappa, plain, typed = flow_only_comparison(7, 1800.0, WINDOW_GRID)
best = int(plain.argmax())
print(f"best window for the plain signed boxcar : {1000 * WINDOW_GRID[best]:.1f} ms"
      f"   (w* beta = {WINDOW_GRID[best] * beta:.2f})")
print(f"  Corr(plain boxcar, kappa)       {plain[best]:.4f}")
print(f"  Corr(type-weighted boxcar, kappa) {typed[typed.argmax()]:.4f}")
print(f"  gap that no window removes:     {typed.max() - plain[best]:.4f}")

best window for the plain signed boxcar : 28.4 ms   (w* beta = 1.70)
  Corr(plain boxcar, kappa)       0.8306
  Corr(type-weighted boxcar, kappa) 0.8705
  gap that no window removes:     0.0398


In [23]:
figure = go.Figure()
figure.add_trace(go.Scatter(x=WINDOW_GRID, y=plain, mode="lines", name="flat weights (OFI's)",
                            line=dict(color=INK)))
figure.add_trace(go.Scatter(x=WINDOW_GRID, y=typed, mode="lines", name="type-weighted",
                            line=dict(color=MUTED, dash="dash")))
figure.update_layout(
    title="A boxcar against the state the intensity reads exponentially",
    xaxis=dict(title="window w (s)", type="log"),
    yaxis=dict(title="Corr( boxcar , kappa )"), height=420,
    legend=dict(x=0.02, y=0.02))
figure.show()

The two curves peak at the same window and never meet. Tuning $w$ closes the *window-shape*
part of the misspecification; a floor remains below it that no window removes, and that floor
is the flat type weighting. It is a result about the estimator, not a defect to be fixed —
$\mathrm{OFI}$ is defined without reference to any kernel, which is exactly why it is usable
on data where the kernel is unknown.

### The ceiling

The oracle is *not* an upper bound on prediction. It is optimal among flow-only predictors,
and $I^n$ reads the book, so it can beat it — which is the content of §10's $\rho = 0$
control.

In [24]:
recorded = RECORDED[CONFIRMATORY[0]]
predictors, outcome, keep, mid = frame_for(recorded)
y = outcome[keep]
classes = ir.outcome_classes(y)
half = y.size // 2

# The oracle needs the Hawkes state behind *this* session's messages, which the message
# stream cannot supply -- the journal is what holds the point process itself.
from unito26.lob.simulate import EventJournal
journal = EventJournal.empty()
simulator = OrderFlowSimulator(FLOW, MARKS, REFERENCE_PRICE, rng=CONFIRMATORY[0])
book = AggregateBook()
simulator.warm_up(book, horizon=WARM_UP, journal=journal)
opening, messages = book.copy(), []
for message in simulator.stream(book, horizon=SAMPLE, journal=journal):
    book.apply(message, record=False)
    messages.append(message)

event_times = np.array(journal.times)
event_kinds = np.array(journal.types)
oracle_kappa = intensities_at_events(FLOW, event_times, event_kinds) @ PRESSURE
emitted = np.array(journal.emitted)
print(f"journal holds {len(journal):,} events; {journal.dropped} were dropped on an empty side,")
print(f"so the message stream carries {emitted.sum():,} -- a strictly coarser filtration.")

journal holds 254,271 events; 6 were dropped on an empty side,
so the message stream carries 254,265 -- a strictly coarser filtration.


In [25]:
# Align the oracle to the session's rows: the messages are the emitted events, in order.
sample_rows = event_times > WARM_UP
kappa_at_message = oracle_kappa[sample_rows & emitted]
offset = len(mid.values) - len(kappa_at_message)
kappa_aligned = np.concatenate([np.full(offset, np.nan), kappa_at_message])[: len(mid.values)]

rows = []
for name, values in (("OFI", predictors.order_flow_imbalance),
                     ("type-weighted boxcar", None),
                     ("flow oracle sign(kappa)", kappa_aligned),
                     ("I^1 (reads the book)", predictors.queue_imbalance)):
    if values is None:
        continue
    x = values[keep]
    usable = np.isfinite(x)
    if usable.sum() < 100:
        continue
    rows.append({
        "predictor": name,
        "rows": int(usable.sum()),
        "Cov(sign, dP)": ir.signed_covariance(x[usable], y[usable]),
        "log-score skill": est.log_score_skill(
            x[usable][: usable.sum() // 2], classes[usable][: usable.sum() // 2],
            x[usable][usable.sum() // 2 :], classes[usable][usable.sum() // 2 :], BINS, PRIOR),
        "tau-b": est.kendall_tau_b(x[usable], y[usable]),
    })
print("The flow-only oracle is built from the generator's own state and still loses to a")
print("statistic read off the book.  A Hawkes simulator does not make flow statistics")
print("optimal by construction, which is the misreading this row exists to prevent.")
pd.DataFrame(rows).round(5)

The flow-only oracle is built from the generator's own state and still loses to a
statistic read off the book.  A Hawkes simulator does not make flow statistics
optimal by construction, which is the misreading this row exists to prevent.


,predictor,rows,"Cov(sign, dP)",log-score skill,tau-b
0,OFI,109117,0.00145,0.00351,0.01682
1,flow oracle sign(kappa),109117,0.00142,0.00227,0.01509
2,I^1 (reads the book),109117,0.01075,0.02607,0.08746


## 9. The $\beta$ ladder, run twice

The excitation half-life is the one timescale in the lab, and the temptation is to run a
single ladder in $\beta$ at a fixed window. That confounds two things. With $w$ fixed, the
aggregation group $\nu w$ is constant while $\nu/\beta$ moves; with $w \propto 1/\beta$, the
kernel group is constant while $\nu w$ spans a factor of $33$ across the ladder. A single
ladder cannot separate a kernel effect from a window effect, and its terminal rung measures
the window.

So the ladder is run **twice**, at fixed $w$ and at fixed $\nu w$, with both dimensionless
groups reported in every row. The pair is the point, not a robustness check.

$\lambda^*$ is held identical on every rung by $\mu = (I - \Gamma)\lambda^*$, so the message
rate and the book's composition do not move with $\beta$.

In [26]:
def rung(branching_ratio, decay, cross=1.0):
    # One ladder rung, holding lambda* and therefore the flow's composition.
    shape = with_cross_pressure_scaled(A, PRESSURE, cross) if cross != 1.0 else A
    return HawkesParams.from_stationary_intensity(lam_star, shape, decay, branching_ratio)


def measure(flow, seed, window, levels=GridDepth(1)):
    # The incremental statistics on one session at one rung.
    recorded = session_at(flow, MARKS, seed)
    mid = ir.aligned_mid_price(recorded)
    predictors = ir.predictors(recorded, window, levels)
    outcome = ir.forward_change_to_next_event(mid)
    keep = predictors.defined & np.isfinite(outcome)
    y = outcome[keep]
    classes = ir.outcome_classes(y)
    half = y.size // 2
    ofi = predictors.order_flow_imbalance[keep]
    last = predictors.last_contribution[keep]
    imbalance = predictors.queue_imbalance[keep]
    return {
        "rows": int(keep.sum()),
        "surviving": float(keep.mean()),
        "segments": int(mid.segments.max()) + 1,
        "skill OFI": est.log_score_skill(
            ofi[:half], classes[:half], ofi[half:], classes[half:], BINS, PRIOR),
        "skill I^1": est.log_score_skill(
            imbalance[:half], classes[:half], imbalance[half:], classes[half:], BINS, PRIOR),
        "inc(window | e_n)": est.incremental_log_score_skill(
            last[:half], (ofi - last)[:half], classes[:half],
            last[half:], (ofi - last)[half:], classes[half:], BINS, PRIOR),
        "inc(OFI | I^1)": est.incremental_log_score_skill(
            imbalance[:half], ofi[:half], classes[:half],
            imbalance[half:], ofi[half:], classes[half:], BINS, PRIOR),
        "Cov(sign OFI, dP)": ir.signed_covariance(ofi, y),
        "atom": float(np.mean(y == 0)),
    }

In [27]:
#: Two or three seeds a rung reads the shape; the pre-registered claims use twelve.
LADDER_SEEDS = list(CONFIRMATORY)[:3]
BETA_RUNGS = (30.0, 60.0, 120.0)
FIXED_WINDOW = W_STAR

rows = []
for decay_rung in BETA_RUNGS:
    flow = rung(FLOW.branching_ratio, decay_rung)
    for label, window in (("fixed w", FIXED_WINDOW),
                          ("fixed nu*w", Window(FIXED_WINDOW * 60.0 / decay_rung))):
        got = [measure(flow, seed, window) for seed in LADDER_SEEDS]
        rows.append({
            "beta": decay_rung, "held": label, "w (ms)": 1000 * window,
            "nu*w": nu * window, "nu/beta": nu / decay_rung,
            **{k: np.mean([g[k] for g in got])
               for k in ("inc(window | e_n)", "inc(OFI | I^1)", "skill OFI", "skill I^1", "atom")},
        })
BETA_LADDER = pd.DataFrame(rows)
print("Both groups are in every row, so a reader can see which one moved.")
BETA_LADDER.round(5)

Both groups are in every row, so a reader can see which one moved.


,beta,held,w (ms),nu*w,nu/beta,inc(window | e_n),inc(OFI | I^1),skill OFI,skill I^1,atom
0,30.0,fixed w,30.0,0.90564,1.00626,0.00227,0.00329,0.00366,0.02868,0.95996
1,30.0,fixed nu*w,60.0,1.81127,1.00626,0.00188,0.00271,0.00259,0.02866,0.95996
2,60.0,fixed w,30.0,0.90564,0.50313,0.00285,0.00564,0.00432,0.02947,0.95874
3,60.0,fixed nu*w,30.0,0.90564,0.50313,0.00285,0.00564,0.00432,0.02947,0.95874
4,120.0,fixed w,30.0,0.90564,0.25157,0.00285,0.00641,0.00416,0.02735,0.95717
5,120.0,fixed nu*w,15.0,0.45282,0.25157,0.00271,0.00683,0.00524,0.02735,0.95717


## 10. The $\rho$ ladder, and the control that carries it

$\rho$ is the amount of endogeneity. The rung at $\rho = 0$ is a Poisson flow with no
clustering at all, and it is the control that makes the whole ladder mean something.

Holding the *total* rate across rungs is not enough. The excitation is not
composition-neutral — market orders are far more excited than limit orders — so raising
$\rho$ with $\mu$ fixed starves the book of market orders and freezes the mid. Every rung
therefore holds $\lambda^*$ **identical**, with $\mu = (I - \Gamma)\lambda^*$ recomputed and
its non-negativity checked rung by rung.

The claim the section makes is the **incremental** one. $\mathrm{OFI}$'s power decomposes
into two channels:

* a **book-state channel**, which survives at $\rho = 0$, where $\mathrm{OFI}$ is a noisy
  reading of the same book $I^n$ reads, and loses to it;
* a **flow-clustering channel**, which exists only if there is a kernel.

So the incremental skill of $\mathrm{OFI}$ over $I^1$ *is* a measurement of the kernel, and
it is the quantity that replicates. The levels do not: they carry $1/\mathrm{Var}(\text{depth})$,
and the depth is not controlled across rungs, so the contemporaneous column is not expected
flat and is not measured flat.

In [28]:
RHO_RUNGS = (0.0, 0.2, 0.4, 0.6, 0.8)
rows = []
for rho_rung in RHO_RUNGS:
    flow = (HawkesParams(baseline=lam_star, excitation=np.zeros((6, 6)), decay=beta)
            if rho_rung == 0 else rung(rho_rung, beta))
    assert (flow.baseline >= 0).all(), f"mu is infeasible at rho = {rho_rung}"
    got = [measure(flow, seed, W_STAR) for seed in LADDER_SEEDS]
    rows.append({
        "rho": rho_rung,
        "min mu": float(flow.baseline.min()),
        "endogenous": flow.endogenous_fraction(),
        "signed endogenous": flow.signed_endogenous_fraction(PRESSURE),
        "cluster size": flow.mean_cluster_size(),
        **{k: np.mean([g[k] for g in got])
           for k in ("inc(OFI | I^1)", "inc(window | e_n)", "skill OFI", "skill I^1", "atom")},
        "seed spread of inc(OFI|I^1)":
            np.std([g["inc(OFI | I^1)"] for g in got], ddof=1) if len(LADDER_SEEDS) > 1 else np.nan,
    })
RHO_LADDER = pd.DataFrame(rows)
print("At rho = 0 there is no kernel, so inc(OFI | I^1) has nothing to measure and I^1")
print("should dominate; the incremental column is the one predicted to rise with rho.")
RHO_LADDER.round(5)

At rho = 0 there is no kernel, so inc(OFI | I^1) has nothing to measure and I^1
should dominate; the incremental column is the one predicted to rise with rho.


,rho,min mu,endogenous,signed endogenous,cluster size,inc(OFI | I^1),inc(window | e_n),skill OFI,skill I^1,atom,seed spread of inc(OFI|I^1)
0,0.0,2.19789,0.00000,0.00000,1.00000,-0.00143,0.00015,0.00074,0.02651,0.95279,0.00026
1,0.2,1.72287,0.19193,0.10099,1.23751,0.00052,0.00036,0.00090,0.02873,0.95141,0.00112
2,0.4,1.24786,0.38385,0.20198,1.62299,0.00310,0.00050,0.00286,0.02633,0.95747,0.00086
3,0.6,0.77285,0.57578,0.30298,2.35727,0.00564,0.00285,0.00432,0.02947,0.95874,0.00083
4,0.8,0.29784,0.76771,0.40397,4.30490,0.00626,0.00554,0.00514,0.02719,0.96233,0.00308


In [29]:
figure = go.Figure()
for column, colour, dash in (("inc(OFI | I^1)", INK, None), ("inc(window | e_n)", MUTED, "dash")):
    figure.add_trace(go.Scatter(x=RHO_LADDER["rho"], y=RHO_LADDER[column],
                                mode="lines+markers", name=column,
                                line=dict(color=colour, dash=dash)))
figure.add_hline(y=0.0, line=dict(color=GRID))
figure.update_layout(title="Incremental skill against the branching ratio",
                     xaxis=dict(title="rho"), yaxis=dict(title="incremental log-score skill"),
                     height=420, legend=dict(x=0.02, y=0.98))
figure.show()

## 11. The $c$ ladder: signed endogeneity at constant total branching

The third ladder varies *what kind* of endogeneity there is, holding how much. Scaling the
opposite-pressure entries of $A$ by $c$ and rescaling back to the same $\rho$ moves the
**signed** endogenous fraction while the total branching stays put.

Two errors to avoid, and the code avoids both by construction. The signed quantity is **not
a spectral radius**: with $P = \mathrm{diag}(p)$ and $P^2 = I$, $P\Gamma P$ is similar to
$\Gamma$ and has exactly its spectrum, so nothing signed lives in the eigenvalues. And the
descendant count is never $1/(1 - \rho_{\text{signed}})$, which would sit *below* the $c = 1$
rung — the same error the corrections commit fixed in the unsigned case.

The internal check is the $c = 0$ rung: there every offspring inherits its parent's sign, so
the signed and unsigned endogenous fractions must coincide **exactly**. This is also the safe
ladder — every rung is feasible, and the slack in $\mu$ *increases* as $c$ falls.

In [30]:
rows = []
for cross in (1.0, 0.75, 0.5, 0.25, 0.0):
    flow = rung(FLOW.branching_ratio, beta, cross=cross)
    got = [measure(flow, seed, W_STAR) for seed in LADDER_SEEDS]
    rows.append({
        "c": cross,
        "rho": flow.branching_ratio,
        "min mu": float(flow.baseline.min()),
        "endogenous": flow.endogenous_fraction(),
        "signed endogenous": flow.signed_endogenous_fraction(PRESSURE),
        "signed descendants": flow.signed_descendants(PRESSURE),
        **{k: np.mean([g[k] for g in got])
           for k in ("inc(OFI | I^1)", "skill OFI", "skill I^1")},
    })
C_LADDER = pd.DataFrame(rows)
last = C_LADDER.iloc[-1]
assert abs(last["signed endogenous"] - last["endogenous"]) < 1e-9, \
    "at c = 0 every offspring inherits its parent's sign, so the two must coincide"
assert np.allclose(C_LADDER["rho"], FLOW.branching_ratio), "total branching is not held"
assert C_LADDER["min mu"].is_monotonic_increasing, "the mu slack should grow as c falls"
print("rho held to machine precision; the signed fraction rises to meet the unsigned one")
print("at c = 0; I^1 is flat because it reads a state rather than a flow.")
C_LADDER.round(5)

/home/claudio/projects/unito26/unito26/lob/imbalance_regression.py:226: RuntimeWarning: invalid value encountered in divide
  mean_depth = np.where(rows > 0, rolling_sum(start, np.nan_to_num(depth)) / (2 * rows), np.nan)


rho held to machine precision; the signed fraction rises to meet the unsigned one
at c = 0; I^1 is flat because it reads a state rather than a flow.


,c,rho,min mu,endogenous,signed endogenous,signed descendants,inc(OFI | I^1),skill OFI,skill I^1
0,1.00,0.6,0.77285,0.57578,0.30298,1.42872,0.00564,0.00432,0.02947
1,0.75,0.6,0.79460,0.57959,0.36067,1.55617,0.00702,0.00496,0.02649
2,0.50,0.6,0.81938,0.58407,0.42711,1.73638,0.00794,0.00629,0.02391
3,0.25,0.6,0.84783,0.58944,0.50454,2.01114,0.01674,0.01165,0.02467
4,0.00,0.6,0.88075,0.59605,0.59605,2.48268,0.03042,0.02248,0.02451


In [31]:
figure = go.Figure()
figure.add_trace(go.Scatter(x=C_LADDER["signed endogenous"], y=C_LADDER["inc(OFI | I^1)"],
                            mode="lines+markers", name="inc(OFI | I^1)", line=dict(color=INK)))
figure.add_trace(go.Scatter(x=C_LADDER["signed endogenous"], y=C_LADDER["skill I^1"],
                            mode="lines+markers", name="skill I^1",
                            line=dict(color=MUTED, dash="dash")))
figure.update_layout(
    title="Plotted against the signed endogenous fraction, not against c",
    xaxis=dict(title="signed endogenous fraction"), yaxis=dict(title="log-score skill"),
    height=420, legend=dict(x=0.02, y=0.98))
figure.show()

## 12. Tuning the window

The theory says the optimum sits at $w^* = c/\beta$, matching a boxcar to the exponential
kernel — and specifically **not** at $c/(\beta(1-\rho))$, which is the cluster relaxation
rather than the kernel. The test is therefore a two-exponent regression,
$$\log w^* = a + b_1 \log(1/\beta) + b_2 \log\!\big(1/(1-\rho)\big),$$
with predicted exponents $(1, 0)$. Regressing on $\log[1/(\beta(1-\rho))]$ across the ladders
cannot falsify anything: the $\beta$ ladder holds $\rho$ fixed, so both hypotheses give slope
one there, and only the $\rho$ ladder separates them.

Two design points. **Tune on one block of seeds and evaluate on a disjoint block** — this is
a simulation with an unlimited supply of independent paths, so same-path cross-validation is
the wrong tool: no purge removes the dependence that matters, which is the *state*. The book's
depth mean-reverts over seconds and every predictor reads it, and the dependence is longer for
longer $w$, biasing selection toward long windows exactly as the purge was meant to prevent.
Where folds are used within a path at all, the purge is $w_{\max} + h$ — the **sum**, since a
row at $t$ reads $(t-w,\, t+h]$ — and it is set by the largest candidate, not the one being
scored.

And **report the whole curve**, not an argmax. The surface is quadratically flat near its
optimum, so $\hat w^*$ is noisy and an argmax regression is weak on top of being censored
where the optimum leaves the admissible band. The primary evidence for the scaling law is a
**collapse test**: plot the criterion against $\log w - \log(1/\beta)$ and ask whether the
rungs superimpose. That uses every grid point.

The contemporaneous criterion is refused outright: its $R^2$ decreases monotonically in $w$,
so its argmax is the smallest point of any grid, whatever the kernel.

In [32]:
TUNE_GRID = np.exp(np.linspace(np.log(0.002), np.log(0.4), 24))

curves = {}
for label, rho_rung, decay_rung in (("rho 0.6, beta 60", 0.6, 60.0),
                                    ("rho 0.6, beta 120", 0.6, 120.0),
                                    ("rho 0.8, beta 60", 0.8, 60.0),
                                    ("rho 0.4, beta 60", 0.4, 60.0)):
    flow = rung(rho_rung, decay_rung)
    _, _, kappa_curve, plain_curve, _ = flow_only_comparison(21, 900.0, TUNE_GRID)
    # rebuild against this rung's own kappa
    simulator = ExponentialHawkes(flow, rng=21)
    events = list(simulator.events(900.0))
    t = np.array([a for a, _ in events]); k = np.array([b for _, b in events])
    kappa_here = intensities_at_events(flow, t, k) @ PRESSURE
    signs = PRESSURE[k]
    def boxcar(values, window, times=t):
        start = np.searchsorted(times, times - window, side="right")
        cumulative = np.concatenate([[0.0], np.cumsum(values)])
        return cumulative[1:] - cumulative[start]
    curve = np.array([np.corrcoef(boxcar(signs, w), kappa_here)[0, 1] for w in TUNE_GRID])
    curves[label] = (curve, decay_rung, rho_rung, TUNE_GRID[curve.argmax()])

for label, (curve, decay_rung, rho_rung, best) in curves.items():
    print(f"{label:20s}  w* = {1000 * best:6.2f} ms   w* beta = {best * decay_rung:.3f}"
          f"   w* beta (1-rho) = {best * decay_rung * (1 - rho_rung):.3f}")

rho 0.6, beta 60      w* =  31.74 ms   w* beta = 1.904   w* beta (1-rho) = 0.762
rho 0.6, beta 120     w* =  15.90 ms   w* beta = 1.908   w* beta (1-rho) = 0.763
rho 0.8, beta 60      w* =  25.21 ms   w* beta = 1.512   w* beta (1-rho) = 0.302
rho 0.4, beta 60      w* =  31.74 ms   w* beta = 1.904   w* beta (1-rho) = 1.143


In [33]:
figure = go.Figure()
for label, (curve, decay_rung, rho_rung, best) in curves.items():
    figure.add_trace(go.Scatter(x=np.log(TUNE_GRID * decay_rung), y=curve,
                                mode="lines", name=label))
figure.update_layout(
    title="The collapse test: the criterion against log(w) - log(1/beta)",
    xaxis=dict(title="log(w * beta)"), yaxis=dict(title="Corr( boxcar , kappa )"),
    height=440, legend=dict(x=0.02, y=0.02))
figure.show()

In [34]:
design = np.column_stack([
    np.ones(len(curves)),
    [np.log(1 / decay_rung) for _, decay_rung, _, _ in curves.values()],
    [np.log(1 / (1 - rho_rung)) for _, _, rho_rung, _ in curves.values()],
])
fitted = est.ols(design, np.log([best for _, _, _, best in curves.values()]))
print(f"exponent on log(1/beta)      {fitted.coefficients[1]:+.3f}   predicted +1")
print(f"exponent on log(1/(1-rho))   {fitted.coefficients[2]:+.3f}   predicted  0")
print()
print("Reported as an equivalence test against a stated margin, not as a test that the")
print("slope differs from zero: the prediction is a point, so the burden runs the other way.")
print("Four rungs at one seed is a shape, not an interval -- the pre-registered version")
print("uses a disjoint seed block and reports the band.")

exponent on log(1/beta)      +0.917   predicted +1
exponent on log(1/(1-rho))   -0.223   predicted  0

Reported as an equivalence test against a stated margin, not as a test that the
slope differs from zero: the prediction is a point, so the burden runs the other way.
Four rungs at one seed is a shape, not an interval -- the pre-registered version
uses a disjoint seed block and reports the band.


## 13. The central experiment, and how it is tested

The question the study exists to answer is whether the *window* adds information over the
last event alone. It is asked twice, and the lattice-aware form is the confirmatory one.

**Confirmatory.** The incremental three-class log-score skill of the binned
$(e_n,\ \mathrm{OFI}_{t,w} - e_n)$ model over the binned $e_n$ model. The joint model is
shrunk toward the $e_n$-only model, so the statistic is exactly zero when the window adds
nothing and needs no separate null.

**Diagnostic.** $\Delta P^m_{(t,t+h]} = a + b_1 e_n + b_2(\mathrm{OFI}_{t,w} - e_n) +
\varepsilon$, testing $b_2 = 0$.

The OLS version cannot be the headline, for two reasons that are worth separating.
$\mathrm{OFI} - e_n$ is a **reparametrisation, not an orthogonalisation** —
$\mathrm{Cov}(e_n, \sum_{k<n} e_k) > 0$ under clustering — so by Frisch–Waugh the *test* on
$b_2$ is the right test, while $b_1$ is not "the $e_n$ effect" and the specification is not
scale-invariant at the coefficient level. And $b_2 = 0$ is a one-degree-of-freedom **linear**
restriction that forces the window's contribution onto an equally weighted boxcar, so
$b_2 \approx 0$ is entirely consistent with the window carrying information the boxcar cannot
express. The honest multi-df version splits the window into lag buckets and takes the joint
$F$; its *shape* also measures the boxcar-versus-exponential mismatch directly.

In [35]:
LAG = 20        # HAC truncation, set from the horizon and not from the window: see below

rows = []
for seed in CONFIRMATORY:
    recorded = RECORDED[seed]
    predictors, outcome, keep, mid = frame_for(recorded)
    y = outcome[keep]
    classes = ir.outcome_classes(y)
    half = y.size // 2
    ofi = predictors.order_flow_imbalance[keep]
    last = predictors.last_contribution[keep]
    imbalance = predictors.queue_imbalance[keep]

    fit = ir.nested_fit(last, ofi, y, lag=LAG)
    buckets = ir.lag_buckets(ir.aligned_order_flow(recorded), W_STAR, 4)[keep]
    statistic, degrees = ir.joint_f(
        np.column_stack([np.ones(y.size), buckets]), y, slice(1, 5), LAG)
    rows.append({
        "seed": seed, "rows": y.size, "surviving": float(keep.mean()),
        "inc(window | e_n)": est.incremental_log_score_skill(
            last[:half], (ofi - last)[:half], classes[:half],
            last[half:], (ofi - last)[half:], classes[half:], BINS, PRIOR),
        "inc(OFI | I^1)": est.incremental_log_score_skill(
            imbalance[:half], ofi[:half], classes[:half],
            imbalance[half:], ofi[half:], classes[half:], BINS, PRIOR),
        "inc(I^1 | OFI)": est.incremental_log_score_skill(
            ofi[:half], imbalance[:half], classes[:half],
            ofi[half:], imbalance[half:], classes[half:], BINS, PRIOR),
        "b1": fit.last, "b2": fit.window, "t(b2)": fit.window / fit.window_error,
        "bucket F": statistic, "df": degrees,
    })
CENTRAL = pd.DataFrame(rows)
CENTRAL.round(6)

,seed,rows,surviving,inc(window | e_n),inc(OFI | I^1),inc(I^1 | OFI),b1,b2,t(b2),bucket F,df
0,1000,109117,0.999973,0.004015,0.004786,0.027372,0.000030,0.000006,1.392366,2.916759,4
1,1001,107963,0.999972,0.002724,0.006443,0.032646,0.000042,0.000012,2.829217,10.465362,4
2,1002,109696,0.999973,0.001812,0.005698,0.032385,0.000042,0.000008,1.759430,6.973594,4
3,1003,107957,0.999972,0.002189,0.002457,0.021235,0.000035,0.000009,2.114223,4.859096,4
4,1004,108024,0.999954,0.003199,0.005415,0.028492,0.000037,0.000012,2.393969,7.609809,4
5,1005,109534,0.999982,0.000881,0.003733,0.028334,0.000046,0.000009,2.163804,10.892032,4
6,1006,109072,0.999982,0.005203,0.007181,0.031042,0.000026,0.000022,4.486522,7.935081,4
7,1007,110005,0.999918,0.003257,0.004359,0.029051,0.000065,0.000017,3.163691,12.168532,4
8,1008,109598,0.999973,0.001385,0.001956,0.025172,0.000028,0.000000,0.130069,3.511954,4
9,1009,108120,0.999972,0.003548,0.005702,0.026114,0.000039,0.000018,3.651778,8.564330,4


In [36]:
def seed_level(values, label):
    values = np.asarray(values, dtype=float)
    n = values.size
    mean, error = values.mean(), values.std(ddof=1) / np.sqrt(n)
    return {"statistic": label, "seeds": n, "mean": mean, "standard error": error,
            "t": mean / error, "95% low": mean - 2.2 * error, "95% high": mean + 2.2 * error}


print("Seed-level intervals.  The seeds are the independent replicates, so every claim")
print("about the *generator* takes this interval and not the within-path bootstrap, which")
print("would be both the wrong estimand and optimistically narrow.")
pd.DataFrame([
    seed_level(CENTRAL["inc(window | e_n)"], "C1  window adds over e_n"),
    seed_level(CENTRAL["inc(OFI | I^1)"], "C2  OFI adds over I^1"),
    seed_level(CENTRAL["inc(I^1 | OFI)"] - CENTRAL["inc(OFI | I^1)"], "C3  I^1 adds more than OFI"),
    seed_level(CENTRAL["t(b2)"], "     t(b2), per seed"),
    seed_level(CENTRAL["bucket F"], "     lag-bucket F, per seed"),
]).round(6)

Seed-level intervals.  The seeds are the independent replicates, so every claim
about the *generator* takes this interval and not the within-path bootstrap, which
would be both the wrong estimand and optimistically narrow.


,statistic,seeds,mean,standard error,t,95% low,95% high
0,C1 window adds over e_n,12,0.002763,0.000350,7.894285,0.001993,0.003533
1,C2 OFI adds over I^1,12,0.004590,0.000466,9.841113,0.003564,0.005616
2,C3 I^1 adds more than OFI,12,0.023299,0.000658,35.419067,0.021851,0.024746
3,"t(b2), per seed",12,2.658452,0.366970,7.244329,1.851118,3.465786
4,"lag-bucket F, per seed",12,7.671751,0.871471,8.803222,5.754515,9.588986


### Why the seeds are unpaired, and what that costs

`OrderFlowSimulator` hands **one** `Generator` to `ExponentialHawkes`, so Hawkes draws and
mark draws interleave on a single stream. Change $\beta$, $\rho$ or $c$ and the first waiting
time changes, after which every later draw desynchronises. The number of mark draws per event
depends on book state too — `_withdrawal` returns early on an empty side, consuming nothing.

So rungs are **independent replicates**, $\mathrm{Var}(\text{difference}) = \mathrm{Var}_1 +
\mathrm{Var}_2$, and no common-random-numbers reduction is available without building one.
Separate `SeedSequence` streams would give it, and would change every recorded run in
`dev-context/`; the study accepts the wider intervals instead. That is a choice, and it is
recorded as one.

### Two levels of uncertainty

The **block bootstrap** gives a within-path interval and is confined to single-session
statements. It resamples contiguous blocks of *time*, not blocks of rows: rows are
event-sampled and clustered, so a fixed row count maps to a wildly variable duration, and it
is the duration that has to exceed the dependence length. The block length comes from
Politis–White on the actual scored series — dominated by the depth process, which is
autocorrelated over *seconds*, where the kernel relaxation of $1/(\beta(1-\rho)) = 42$ ms is
under two events.

**HAC** backs it. The truncation lag is set from the forecast horizon $h$, not from
$\max(w, h)$: under a correctly specified conditional mean the score $x_t\varepsilon_t$
inherits its serial correlation from the *forecast overlap*, and a long lookback makes the
regressor persistent without by itself making the score autocorrelated. The plug-in
$4(n/100)^{2/9}$ is blind to that — 12 rows at $n = 14{,}000$ and 21 at $n = 180{,}000$,
whatever the window.

In [37]:
recorded = RECORDED[CONFIRMATORY[0]]
predictors, outcome, keep, mid = frame_for(recorded)
y = outcome[keep]
x = predictors.order_flow_imbalance[keep]
times = mid.times[keep]

fit = est.ols(np.column_stack([np.ones(y.size), x]), y)
print("HAC sensitivity, reported at L, 2L and L/2 rather than at one lag:")
for lag in (LAG // 2, LAG, 2 * LAG):
    robust = np.sqrt(est.hac_variance(fit, lag)[1, 1])
    print(f"  L = {lag:3d}   se(slope) {robust:.4e}"
          f"   effective n {est.effective_sample(fit, lag, 1):,.0f} of {y.size:,}")
print()
print("effective_n is a property of *that coefficient*, not of the row set, and it differs")
print("across the columns of one fit.  It is descriptive; the intervals come from seeds.")

HAC sensitivity, reported at L, 2L and L/2 rather than at one lag:
  L =  10   se(slope) 4.0260e-06   effective n 95,769 of 109,117
  L =  20   se(slope) 3.9975e-06   effective n 97,140 of 109,117
  L =  40   se(slope) 3.9910e-06   effective n 97,456 of 109,117

effective_n is a property of *that coefficient*, not of the row set, and it differs
across the columns of one fit.  It is descriptive; the intervals come from seeds.


In [38]:
scores = ir.signed_covariance_terms(x, y)
block_rows = est.politis_white_block_length((scores[:, 0] - scores[:, 0].mean()))
span = float(np.median(np.diff(times)))
print(f"Politis-White block length: {block_rows:.1f} rows")
print(f"median inter-row gap      : {1000 * span:.2f} ms")
print()
print("Sensitivity of the within-path interval to the block duration:")
for factor in (1.0, 2.0, 4.0):
    duration = factor * max(block_rows * span, 0.5)
    means = est.bootstrap_means(times, scores, duration, 200, np.random.default_rng(3))
    draws = means[:, 0] - means[:, 1] * means[:, 2]
    print(f"  block {duration:6.2f}s   Cov(sign, dP) = {ir.signed_covariance(x, y):+.6f}"
          f"   95% [{np.quantile(draws, 0.025):+.6f}, {np.quantile(draws, 0.975):+.6f}]")
print()
print("An interval sensitive to the block length is not quoted; one insensitive to it is.")

Politis-White block length: 9.0 rows
median inter-row gap      : 10.36 ms

Sensitivity of the within-path interval to the block duration:


  block   0.50s   Cov(sign, dP) = +0.001452   95% [+0.000319, +0.001791]


  block   1.00s   Cov(sign, dP) = +0.001452   95% [+0.000297, +0.001574]


  block   2.00s   Cov(sign, dP) = +0.001452   95% [+0.000475, +0.001780]

An interval sensitive to the block length is not quoted; one insensitive to it is.


### What the row set costs, and the audit

The masking is worth an explicit accounting, because this study had to discard its own
exploratory numbers over exactly this. An earlier pass reported a participation ratio of 46,
96.7% of $\sum(\Delta P^m)^2$ in the top 0.1%, and a 94.9% atom — all measured *before* the
alignment mask, and every one of them moved in the direction the bug predicts once masked.

The lesson is not "be careful". It is that an alignment error in this setting does not raise
and does not look wrong: it makes the fit *better*, because it leaks the contemporaneous
relation into the predictive one.

In [39]:
audit = []
recorded = RECORDED[CONFIRMATORY[0]]
mid = ir.aligned_mid_price(recorded)
flow = ir.aligned_order_flow(recorded)
predictors = ir.predictors(recorded, W_STAR, GridDepth(1))
next_event = ir.forward_change_to_next_event(mid)

for label, mask in (
    ("everything defined (used)", predictors.defined & np.isfinite(next_event)),
    ("OFI defined", np.isfinite(predictors.order_flow_imbalance)),
    ("I^1 covered", np.isfinite(predictors.queue_imbalance)),
    ("outcome defined", np.isfinite(next_event)),
):
    audit.append({"row set": label, "rows": int(mask.sum()),
                  "share": float(mask.mean())})
print("The row sets coincide except at the session opening, which is what makes the")
print("comparison in section 5 a comparison rather than two samples.")
pd.DataFrame(audit).round(6)

The row sets coincide except at the session opening, which is what makes the
comparison in section 5 a comparison rather than two samples.


,row set,rows,share
0,everything defined (used),109117,0.999973
1,OFI defined,109118,0.999982
2,I^1 covered,109120,1.000000
3,outcome defined,109119,0.999991


## 14. Findings

**What the window buys.** Over the twelve pre-registered confirmatory seeds, the incremental
log-score skill of $(e_n,\ \mathrm{OFI} - e_n)$ over $e_n$ alone is positive on every seed:
$+2.8\times 10^{-3}$ on average, $t \approx 7.9$ across seeds. The window carries information
the last event does not. The OLS diagnostic agrees more weakly — $t(b_2) \approx 2.7$ per
seed on average — and the four-bucket $F$ agrees strongly, which is the pattern the theory
predicts when the true weighting is exponential and the tested one is a boxcar.

**What the book buys, and it is more.** $\mathrm{OFI}$ adds $+4.6\times 10^{-3}$ of skill
over $I^1$; $I^1$ adds $+2.8\times 10^{-2}$ over $\mathrm{OFI}$ — six times as much, on the
same rows, at the same bin count, in the same folds. The flow-only oracle built from the
generator's own state also loses to $I^1$. A Hawkes simulator does not make flow statistics
optimal by construction, and reading this study as a horse race that $\mathrm{OFI}$ ought to
win is the misreading it is designed to prevent.

**What contradicted prediction.** Three things.

*The book empties, and an earlier draft of this study said it does not.* At a wider mark
offset, six seeds and 654,000 messages produced no empty-side row and were read as "never";
thirty seeds produced twelve that empty. A rare-event property cannot be asserted from a
block sized for a common one. At the operating point the rate is 26 rows in a million and
the twelve confirmatory seeds happen to show none — which is why the gate is a **rate with
a stated tolerance** and the segment mask stays in the code, rather than an assertion of
zero that a larger block would falsify.

*The $\rho = 0$ control behaves exactly as the estimator predicts.* With no kernel there is
nothing for the window to add, and the incremental skill of $\mathrm{OFI}$ over $I^1$ comes
out **negative** — $-1.4\times 10^{-3}$, the finite-sample penalty of a richer model fitted
on noise. It then rises monotonically through the ladder to $+6.3\times 10^{-3}$ at
$\rho = 0.8$. That the control lands on the penalty rather than on zero is the check that
the statistic is calibrated.

*Concentration, non-emptiness and a moving mid are one trade, not three requirements.* With
the spread pinned at a tick the mid moves only by clearing a touch queue, which is also the
first step in emptying a side; only occupied depth *behind* the touch separates them. The
parametrization that best satisfies a top-of-book reading of $\mathrm{OFI}$ is the one that
produces 159-tick single-event moves.

*The composition sits on a boundary.* $\lambda_L/(\lambda_M + \lambda_W) = 1$ is not a
preference but the point where the depth is stationary; on one side the book empties, on the
other $1/(2\bar S)$ stops being a coefficient.

**The limitation, stated last because it governs everything above.** Nothing measured here is
evidence about markets. It is a result about two estimators of a *known* generator, read
against that generator's own dimensionless groups. The generator has no book-reading
intensities, no informed trading, no queue-position economics, one excitation timescale, no
intraday non-stationarity and no anchor for the price level. Whether any regime studied here
occurs in a traded market, and whether one excitation timescale suffices for one, are
questions that cannot be asked without recorded data.

---

### References

Cont, Kukanov and Stoikov (2014), *JFEC* **12**(1):47–88 — $e_n$, $\mathrm{OFI}$, the depth
normalisation, the mechanical relation and the price-changing-event control. Cont, Stoikov
and Talreja (2010), *Operations Research* **58**(3):549–563 — the state-dependent intensity
model this simulator does not implement. Gould and Bonart (2016), *Market Microstructure and
Liquidity* **2**(2) art. 1650006 — a logistic regression of the *direction of the next
mid-price move* on queue imbalance, so the reference for the $I^n$ half and not for
$\Delta P$ on $\mathrm{OFI}$, together with the large-tick/small-tick contrast. Cont,
Cucuringu and Zhang (2023), *Quantitative Finance* **23**(10):1373–1393 — multi-level and
integrated $\mathrm{OFI}$. Cartea, Donnelly and Jaimungal (2018), *AMF* **25**(1):1–35 — what
volume imbalance predicts, and inside which model. Hawkes (1971), *Biometrika* **58**(1);
Hawkes and Oakes (1974), *J. Appl. Prob.* **11**(3):493–503; Daley and Vere-Jones (2003);
Meyer (1971); Papangelou (1972), *Trans. AMS* **165**:483–506; Ogata (1981), *IEEE Trans.
Inf. Theory* **27**(1):23–31 and (1988), *JASA* **83**(401):9–27; Dassios and Zhao (2013),
*ECP* **18**(62):1–13. Politis and White (2004), *Econometric Reviews* **23**(1):53–70 — the
automatic block length. Clark and West (2007), *J. Econometrics* **138**(1):291–311 — the
nested-forecast adjustment.

No quantity measured here is compared with a published estimate.